In [ ]:
# ================================
# 1. Install Dependencies
# ================================
!pip install -q transformers accelerate sentencepiece bitsandbytes flask huggingface_hub requests

# --- Install cloudflared to create a public API endpoint ---
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
# ================================
# 2. Secure Hugging Face Login
# ================================
# Add your token in Colab: left sidebar -> key icon (Secrets) -> name it HF_TOKEN, turn on notebook access.
# NEVER paste the token into the notebook text.
from google.colab import userdata
from huggingface_hub import login

try:
    login(token=userdata.get("HF_TOKEN"))
    print("Successfully logged into Hugging Face.")
except Exception as e:
    print("Hugging Face token not found or invalid.")
    print("Please add your token to Colab Secrets with the name 'HF_TOKEN'.")
    raise e

In [ ]:
# ================================
# 3. Imports & Model Loading
# ================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import threading
import subprocess
from flask import Flask, request, jsonify

# Model ID for Llama 3.1 8B Instruct
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Configure 4-bit quantization to save memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("⏳ Loading model... This will take a few minutes.")
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"  # Automatically use the GPU
)

# Create a text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Define terminators for Llama 3.1 to ensure clean outputs
terminators = [
    pipe.tokenizer.eos_token_id,
    pipe.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("✅ Model loaded successfully!")

In [ ]:
# ================================
# 4. Define and Run the API Server
# ================================
app = Flask(__name__)

@app.route("/query", methods=["POST"])
def query_model():
    data = request.json
    user_prompt = data.get("prompt", "")
    # optional per-request settings (used for prompt experiments); defaults match the original server
    system_prompt = data.get("system") or "You are a helpful AI assistant who provides clear and concise answers."
    temperature = float(data.get("temperature", 0.6))

    if not user_prompt:
        return jsonify({"error": "No prompt provided"}), 400

    # Format the prompt using the official Llama 3.1 chat template
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    # Apply the template to create the final prompt string
    prompt_for_model = pipe.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Generate the response
    outputs = pipe(
        prompt_for_model,
        max_new_tokens=512,
        eos_token_id=terminators,
        do_sample=temperature > 0,
        **({"temperature": temperature, "top_p": 0.9} if temperature > 0 else {}),
    )

    # Extract and clean the generated text
    response_text = outputs[0]["generated_text"][len(prompt_for_model):]
    print(f"\n--- User Prompt ---\n{user_prompt}\n--- Generated Response ---\n{response_text}\n")
    return jsonify({"response": response_text})

def run_cloudflared():
    command = ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5000", "--no-autoupdate"]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "trycloudflare.com" in line:
            print("=======================================================================================")
            print(f"🌍 Your public API endpoint is live: {line.strip()}")
            print("=======================================================================================")
    process.wait()

# Start the tunnel in a separate thread
thread = threading.Thread(target=run_cloudflared, daemon=True)
thread.start()

print("🚀 Starting Flask server...")
app.run(host="0.0.0.0", port=5000)